<a href="https://colab.research.google.com/github/lasigeBioTM/data-text-processing-notebooks/blob/main/04-task-repetition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unix Shell Tutorial: Task Repetition for Biomedical Data Retrieval

This is the **fourth tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to use task repetition techniques to efficiently apply the same task to all proteins in the list we gathered in the previous tutorial. The task will be to construct the URL that will enable us to download the information from UniProt for each protein using the [RESTful web services provided by UniProt](https://www.uniprot.org/help/api).

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 1: Constructing UniProt URLs with xargs

Given a protein identifier, we can construct the URL that will enable us to download its information from UniProt. We can use the RESTful web services provided by UniProt, more specifically the ones that allow us to retrieve a specific entry. The construction of the URL is simple, it always starts with `https://rest.uniprot.org/uniprotkb/`, followed by the protein identifier, and ends with a dot and the data format. For example, the link for protein P21817 using the XML format is:

`https://rest.uniprot.org/uniprotkb/P21817.xml`

### Assembly line

However, we need to construct one URL for each protein from the list we previously retrieved. The size of the list can be large (hundreds of proteins), varies for different compounds, and evolves with time. Thus, we need an assembly line in which a list of protein identifiers, independently of its size, are added as input to commands that construct one URL for each protein and retrieve the respective file.

The `xargs` command line tool works as an assembly line, it executes a command per each line given as input.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the `chebi_27732_xrefs_UniProt_relevant_identifiers.csv` file directly from the GitHub repository:

In [1]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/03-data-extraction-files.zip'
unzip -o 03-data-extraction-files.zip chebi_27732_xrefs_UniProt_relevant_identifiers.csv
rm -f 03-data-extraction-files.zip

Archive:  03-data-extraction-files.zip
 extracting: chebi_27732_xrefs_UniProt_relevant_identifiers.csv  


We can start by experimenting with the `xargs` command by giving as input the list of protein identifiers in file `chebi_27732_xrefs_UniProt_relevant_identifiers.csv`, and display each identifier on the screen in the middle of a text message by providing the echo command as argument:

In [2]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv | xargs -I {} echo 'Another protein id {} to retrieve'

Another protein id E9PZQ0 to retrieve
Another protein id P21817 to retrieve
Another protein id F1LMY4 to retrieve


The `xargs` command received as input the contents of our CSV file, and for each line displayed a message including the identifier in that line. The `-I` option tells `xargs` to replace `{}` in the command line given as argument by the value of the line being processed. The equivalent long form to the `-I` option is `--replace=R`.

Instead of creating inconsequential text messages, we can use xargs to create the URLs:

In [3]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv | xargs -I {} echo 'https://rest.uniprot.org/uniprotkb/{}.xml'

https://rest.uniprot.org/uniprotkb/E9PZQ0.xml
https://rest.uniprot.org/uniprotkb/P21817.xml
https://rest.uniprot.org/uniprotkb/F1LMY4.xml


We can try to use these links in our internet browser to check if those displayed URLs are working correctly.

In the next step, we will automatically download the files using the `curl` command.

## Step 2: Downloading Files with curl

Now that we have the URLs, we can automatically download the files using the `curl` command instead of echo:

In [4]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv | xargs -I {} curl -O 'https://rest.uniprot.org/uniprotkb/{}.xml'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  130k  100  130k    0     0   176k      0 --:--:-- --:--:-- --:--:--  176k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  216k  100  216k    0     0   259k      0 --:--:-- --:--:-- --:--:--  260k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 62894  100 62894    0     0   100k      0 --:--:-- --:--:-- --:--:--  100k


We should note that we now use the `-O` option to save the output to a given file, named after each protein identifier.

To check if everything worked as expected, we can use the `ls` command to view which files were created:

In [5]:
%%bash
ls *.xml

E9PZQ0.xml
F1LMY4.xml
P21817.xml


The asterisk character (`*`) is here used to represent any file whose name ends with `.xml`.

To check the contents of any of them, we can use the `head` command:

In [6]:
%%bash
head P21817.xml

<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>
<uniprot xmlns="http://uniprot.org/uniprot" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://uniprot.org/uniprot http://www.uniprot.org/docs/uniprot.xsd">
<entry dataset="Swiss-Prot" created="1991-05-01" modified="2026-09-02" version="256" xmlns="http://uniprot.org/uniprot">
  <accession>P21817</accession>
  <accession>Q16314</accession>
  <accession>Q16368</accession>
  <accession>Q9NPK1</accession>
  <accession>Q9P1U4</accession>
  <name>RYR1_HUMAN</name>
  <protein>


### File header

We should note that the content of every XML file starts with `<?xml version="1.0" encoding="UTF-8"?>`, which is the standard XML declaration.


In [7]:
%%bash
head -n 1 *.xml

==> E9PZQ0.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>

==> F1LMY4.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>

==> P21817.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>


The `-n` option specifies how many lines to print, in the previous command just one.

If for any reason, we are not able to download the files from UniProt, we can get them from the [book file archive](http://labs.rd.ciencias.ulisboa.pt/book/)

In the next step, we will create a script using `xargs`.

## Step 3: Creating a Reusable Script

We can now update our script file from previous tutorials. First, let's create or edit the script file `getproteins.sh`:

The script should contain the following lines:

In [8]:
%%bash
cat > getproteins.sh << 'EOF'
# CHEBI identifier given as input is renamed to ID
ID=$1

# Removes any previous files
rm -f chebi_${ID}_*.xml

# Creates a variable with the filename
csv_file=chebi_${ID}_xrefs_UniProt_relevant_identifiers.csv

cat "$csv_file" | \
  xargs \
    -I {} curl 'https://rest.uniprot.org/uniprotkb/{}.xml' -o "chebi_${ID}_{}.xml"
EOF



```
# This is formatted as code
```

**Expected Output:** No output, script file created

### Variable

We should note that the last line now includes the `xargs` and `curl` commands, and the `$ID` variable. This new variable is created in the first line to contain the first value given as argument (`$1`). So, every time we mention `$ID` in the script we are mentioning the first value given as argument. This avoids ambiguity in cases where `$1` is used for other purposes. Since the preceding character of `$ID` is an underscore (`_`), we have to add a backslash (`\`) before it. The second line uses the `rm` command to remove any files that were downloaded in a previous execution. We also added two comments (lines starting with `#`), so we humans do not forget why these commands are needed for.

To execute the script once more:

In [9]:
%%bash
chmod u+x getproteins.sh

**Expected Output:** `(No output, permissions changed)`

In [10]:
%%bash
./getproteins.sh 27732

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  130k  100  130k    0     0   184k      0 --:--:-- --:--:-- --:--:--  184k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  216k  100  216k    0     0   267k      0 --:--:-- --:--:-- --:--:--  267k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 62894  100 62894    0     0   102k      0 --:--:-- --:--:-- --:--:--  103k


And again, to check the results:

In [11]:
%%bash
head -n 1 chebi_27732_*.xml

==> chebi_27732_E9PZQ0.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>

==> chebi_27732_F1LMY4.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>

==> chebi_27732_P21817.xml <==
<?xml version="1.0" encoding="UTF-8"  standalone="no" ?>


## Conclusion

This concludes the **Task Repetition** tutorial adapted from the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to use `xargs` to perform task repetition and efficiently download XML files for all selected proteins from UniProt.

The next tutorial in this series will explore XML parsing techniques to extract relevant information about each protein.


## Exercise

As an exercise, execute the script to download the files for the following protein identifiers associated with [water](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15377):

```
A0AJB3
A0KFC0
A0M6M4
A0Q804
A0QNX1
A1ADU3
```

**Hint**: create a file named `chebi_15377_xrefs_UniProt_relevant_identifiers.csv` with those identifiers.

### Solution

First, create the CSV file with the protein identifiers:

In [12]:
%%bash
cat > chebi_15377_xrefs_UniProt_relevant_identifiers.csv << 'EOF'
A0AJB3
A0KFC0
A0M6M4
A0Q804
A0QNX1
A1ADU3
EOF

**Expected Output:** `(No output, file created)`

Then, execute the script with the water CHEBI identifier:

In [13]:
%%bash
./getproteins.sh 15377

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11518  100 11518    0     0  27328      0 --:--:-- --:--:-- --:--:-- 27293
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12467  100 12467    0     0  30095      0 --:--:-- --:--:-- --:--:-- 30113
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10883  100 10883    0     0  26351      0 --:--:-- --:--:-- --:--:-- 26351
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12509  100 12509    0     0  29665      0 --:--:-- --:--:-- --:--:-- 29712
  % Total    % Received % Xferd  Average Speed   Tim

Verify the downloaded files:

In [14]:
%%bash
ls chebi_15377_*.xml

chebi_15377_A0AJB3.xml
chebi_15377_A0KFC0.xml
chebi_15377_A0M6M4.xml
chebi_15377_A0Q804.xml
chebi_15377_A0QNX1.xml
chebi_15377_A1ADU3.xml
